# FTW Test Results — SGA JPG Images

Runs the FTW segmentation model (`weights/iou=0.726.ckpt`) on 5 JPEG test images
from `data/sga-test/`.

## Documented Limitations

These images are **not** Sentinel-2 data. They are single-temporal RGB JPEGs with no
geospatial metadata. To run them through a model that expects 8-channel
(2 temporal × 4-band RGBN) uint16 GeoTIFFs, the following workarounds are applied:

| Issue | Workaround |
|---|---|
| No NIR channel | Green band is duplicated as synthetic NIR |
| Single temporal layer | The 4-band block `[R, G, B, G]` is duplicated → `[R,G,B,G, R,G,B,G]` |
| DN range mismatch | RGB `[0,255]` scaled to `[0,3000]` (Sentinel-2 L2A range) |
| No geospatial metadata | Dummy CRS EPSG:32632 + 10 m pixel size assigned |

**Spatial coordinates and area figures are not real-world meaningful.**
Results show what the model detects in non-Sentinel-2 imagery.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from PIL import Image
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import torch
import geopandas as gpd

# plot_utils lives in the notebooks directory
sys.path.insert(0, os.path.abspath("."))
from plot_utils import morphological_opening, watershed_segmentation

from ftw_tools.inference.inference import run as inference_run
from ftw_tools.postprocess.polygonize import polygonize

In [ ]:
REPO_ROOT      = os.path.abspath("..")
CHECKPOINT     = os.path.join(REPO_ROOT, "weights", "iou=0.726.ckpt")
TEST_DATA_DIR  = os.path.join(REPO_ROOT, "data", "sga-test")
DRONE_DIR      = os.path.join(TEST_DATA_DIR, "drone")
SATELLITE_DIR  = os.path.join(TEST_DATA_DIR, "satellite")
FTW_DIR        = os.path.join(REPO_ROOT, "outputs", "test_7v9677na")
OUTPUT_DIR     = os.path.join(REPO_ROOT, "outputs", "test_jpg_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Images split by sensor type
SGA_GROUPS = {
    "drone":     {"dir": DRONE_DIR,     "files": ["C081_orginal.jpg", "H6_orginal.jpg", "S10_orginal.jpg"]},
    "satellite": {"dir": SATELLITE_DIR, "files": ["demo_area01.jpg", "demo_area02.jpg"]},
}

# Dummy geospatial parameters (EPSG:32632 UTM Zone 32N, 10 m pixels)
# UTM metric CRS lets polygonize() compute area in m² without reprojection.
PIXEL_SIZE_M = 10.0
ORIGIN_X     = 500_000.0
ORIGIN_Y     = 5_400_000.0
TARGET_CRS   = CRS.from_epsg(32632)

CMAP = ListedColormap(["#888888", "#4CAF50", "#F44336"])
LEGEND_PATCHES = [
    mpatches.Patch(color="#888888", label="Background (0)"),
    mpatches.Patch(color="#4CAF50", label="Field (1)"),
    mpatches.Patch(color="#F44336", label="Boundary (2)"),
]

print(f"Checkpoint : {CHECKPOINT}")
print(f"Drone dir  : {DRONE_DIR}")
print(f"Satellite  : {SATELLITE_DIR}")
print(f"FTW dir    : {FTW_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

## 1. Checkpoint Inspection

Load the checkpoint metadata to confirm architecture and channel count before
building the preprocessing pipeline.

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
hparams = ckpt["hyper_parameters"]
print("Checkpoint hyper_parameters:")
for k, v in hparams.items():
    print(f"  {k}: {v}")

IN_CHANNELS = hparams["in_channels"]
NUM_CLASSES = hparams["num_classes"]
print(f"\nin_channels : {IN_CHANNELS}  (expect 8 = 2 temporal × 4 bands RGBN)")
print(f"num_classes : {NUM_CLASSES}  (0=background, 1=field, 2=boundary)")

del ckpt  # free ~1 GB; inference_run() reloads from disk

## 2. FTW vs SGA Sample Comparison

The SGA test set contains two sensor types with different characteristics:

| Property | FTW (Sentinel-2) | SGA Satellite (JPG) | SGA Drone (JPG) |
|---|---|---|---|
| Format | GeoTIFF | JPEG | JPEG |
| Data type | uint16 | uint8 | uint8 |
| Channels | 8 (2× [R,G,B,NIR]) | 3 (RGB only) | 3 (RGB only) |
| Temporal layers | 2 | 1 | 1 |
| Has NIR | Yes | No | No |
| DN range | 0–10 000+ | 0–255 | 0–255 |
| Patch size | 256×256 px fixed | ~3250×3260 px | ~1600×1480 px |
| Geospatial CRS | EPSG:4326 | None | None |
| Pixel size | ~6 m | Unknown | Unknown (sub-metre likely) |

In [ ]:
# --- Load FTW stacked TIFs ---
ftw_fnames = sorted(f for f in os.listdir(FTW_DIR) if f.endswith("_stacked.tif"))
ftw_samples = {}
for fname in ftw_fnames:
    stem = fname.replace("_stacked.tif", "")
    path = os.path.join(FTW_DIR, fname)
    with rasterio.open(path) as src:
        data = src.read()       # (8, H, W) uint16
        res  = src.res          # (x_deg, y_deg)
        crs  = str(src.crs)
    ftw_samples[stem] = {"path": path, "data": data, "res": res, "crs": crs}

# --- Load SGA images split by sensor type ---
sga_groups = {}   # "drone" / "satellite" → {stem: {path, data}}
sga_samples = {}  # all images flat, for pipeline use
for group_name, group_info in SGA_GROUPS.items():
    group_data = {}
    for fname in group_info["files"]:
        stem = os.path.splitext(fname)[0]
        path = os.path.join(group_info["dir"], fname)
        img  = np.array(Image.open(path).convert("RGB"))  # (H, W, 3) uint8
        group_data[stem]  = {"path": path, "data": img, "group": group_name}
        sga_samples[stem] = group_data[stem]
    sga_groups[group_name] = group_data

print(f"FTW samples      : {len(ftw_samples)}  → {list(ftw_samples.keys())}")
for gname, gdata in sga_groups.items():
    print(f"SGA {gname:10s}: {len(gdata)}  → {list(gdata.keys())}")

# --- Metadata comparison table ---
rows = []
for stem, s in ftw_samples.items():
    d = s["data"]
    _, h, w = d.shape
    m_per_px = s["res"][0] * 111_000
    rows.append({"Name": stem, "Source": "FTW (Sentinel-2)", "Sensor": "satellite",
                 "H×W": f"{h}×{w}", "Channels": d.shape[0], "dtype": str(d.dtype),
                 "CRS": s["crs"], "Approx px size": f"~{m_per_px:.0f} m",
                 "DN min": int(d.min()), "DN max": int(d.max()), "DN mean": f"{d.mean():.0f}"})
for gname, gdata in sga_groups.items():
    for stem, s in gdata.items():
        d = s["data"]
        h, w, c = d.shape
        rows.append({"Name": stem, "Source": f"SGA ({gname})", "Sensor": gname,
                     "H×W": f"{h}×{w}", "Channels": c, "dtype": str(d.dtype),
                     "CRS": "None", "Approx px size": "Unknown",
                     "DN min": int(d.min()), "DN max": int(d.max()), "DN mean": f"{d.mean():.0f}"})

display(pd.DataFrame(rows))

In [ ]:
# Visual comparison: FTW T1 RGB vs Drone vs Satellite
# FTW bands 0,1,2 = T1 Red, Green, Blue; normalize by /3000 and clip to [0,1]

def _ftw_rgb(s):
    rgb = s["data"][:3].transpose(1, 2, 0).astype(np.float32) / 3000.0
    return np.clip(rgb, 0, 1)

def _img_title(stem, s, sensor_label):
    d = s["data"]
    if sensor_label == "FTW":
        _, h, w = d.shape
        m = s["res"][0] * 111_000
        return f"{stem}\n{h}×{w} px  uint16\n~{m:.0f} m/px  {s['crs']}"
    else:
        h, w = d.shape[:2]
        return f"{stem}\n{h}×{w} px  uint8\nunk. res  no CRS"

ftw_items = list(ftw_samples.items())

for group_label, group_color in [("drone", "#FF6F00"), ("satellite", "#1565C0")]:
    grp_items = list(sga_groups[group_label].items())
    n_ftw_show = min(len(grp_items), len(ftw_items))  # match column count
    n_cols = max(len(grp_items), n_ftw_show)

    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 9))
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    # Row 0: FTW T1 RGB
    for col in range(n_cols):
        if col < n_ftw_show:
            stem_f, sf = ftw_items[col]
            axes[0, col].imshow(_ftw_rgb(sf))
            axes[0, col].set_title(_img_title(stem_f, sf, "FTW"), fontsize=7)
        axes[0, col].axis("off")

    # Row 1: SGA group
    for col, (stem_s, ss) in enumerate(grp_items):
        axes[1, col].imshow(ss["data"])
        axes[1, col].set_title(_img_title(stem_s, ss, group_label), fontsize=7)
        axes[1, col].axis("off")
    for col in range(len(grp_items), n_cols):
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("FTW T1 RGB\n(Sentinel-2, uint16)", fontsize=9, labelpad=6)
    axes[1, 0].set_ylabel(f"SGA {group_label.title()}\n(JPEG, uint8)",
                          color=group_color, fontsize=9, labelpad=6)
    for ax in axes.flat:
        ax.axis("off")

    plt.suptitle(f"Visual Comparison: FTW T1 RGB  vs  SGA {group_label.title()}",
                 fontsize=12, color=group_color)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"comparison_visual_{group_label}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Dynamic range comparison: FTW T1 bands (R,G,B,NIR) vs Drone vs Satellite
# Three rows; pooled across all samples within each group for stable histograms.

# Pool FTW T1 bands 0–3 → (4, N_pixels)
ftw_pool = np.concatenate(
    [s["data"][:4].reshape(4, -1) for s in ftw_samples.values()], axis=1
)
# Pool per SGA group → (3, N_pixels)
sga_pools = {
    gname: np.concatenate(
        [s["data"].reshape(-1, 3).T for s in gdata.values()], axis=1
    )
    for gname, gdata in sga_groups.items()
}

band_names   = ["T1 Red (B1)",   "T1 Green (B2)", "T1 Blue (B3)", "T1 NIR (B4)"]
band_colors  = ["#E53935",        "#43A047",        "#1E88E5",       "#7B1FA2"]
chan_names   = ["Red",            "Green",          "Blue"]
chan_colors  = ["#E53935",        "#43A047",        "#1E88E5"]
group_colors = {"drone": "#FF6F00", "satellite": "#1565C0"}

fig, axes = plt.subplots(3, 4, figsize=(18, 12))

# --- Row 0: FTW T1 bands ---
for i, (name, color) in enumerate(zip(band_names, band_colors)):
    vals = ftw_pool[i]
    axes[0, i].hist(vals, bins=120, color=color, alpha=0.85, edgecolor="none")
    median = np.median(vals)
    axes[0, i].axvline(median, color="black", linestyle="--", linewidth=1.2,
                       label=f"median={median:.0f}")
    axes[0, i].set_title(f"FTW {name}\nrange: {vals.min()}–{vals.max()}", fontsize=9)
    axes[0, i].set_xlabel("DN (uint16)")
    axes[0, i].set_ylabel("Pixel count")
    axes[0, i].set_xlim(0, 10_000)
    axes[0, i].legend(fontsize=8)

# --- Rows 1–2: SGA drone and satellite ---
for row_idx, (gname, pool) in enumerate(sga_pools.items(), start=1):
    gcol = group_colors[gname]
    for i, (name, _) in enumerate(zip(chan_names, chan_colors)):
        vals = pool[i]
        axes[row_idx, i].hist(vals, bins=64, color=gcol, alpha=0.75, edgecolor="none")
        median = np.median(vals)
        axes[row_idx, i].axvline(median, color="black", linestyle="--", linewidth=1.2,
                                 label=f"median={median:.0f}")
        axes[row_idx, i].set_title(f"SGA {gname.title()} {name}\nrange: {vals.min()}–{vals.max()}",
                                   fontsize=9, color=gcol)
        axes[row_idx, i].set_xlabel("DN (uint8)")
        axes[row_idx, i].set_ylabel("Pixel count")
        axes[row_idx, i].set_xlim(0, 255)
        axes[row_idx, i].legend(fontsize=8)

    # 4th column: NIR absent + summary note
    ftw_nir_med = np.median(ftw_pool[3])
    ftw_rgb_med = np.median(ftw_pool[:3])
    sga_med     = np.median(pool)
    axes[row_idx, 3].axis("off")
    axes[row_idx, 3].text(
        0.5, 0.5,
        f"NIR unavailable in JPEG format.\n"
        f"Green will be duplicated as\nsynthetic NIR in Section 3.\n\n"
        f"FTW NIR (B4) median ≈ {ftw_nir_med:.0f}\n"
        f"FTW RGB median   ≈ {ftw_rgb_med:.0f}\n"
        f"(uint16, 0–10 000 range)\n\n"
        f"SGA {gname.title()} RGB median ≈ {sga_med:.0f}\n"
        f"(uint8, 0–255 range)",
        ha="center", va="center", transform=axes[row_idx, 3].transAxes,
        fontsize=9, color=gcol,
        bbox=dict(facecolor="#f9f9f9", edgecolor=gcol, boxstyle="round,pad=0.6")
    )
    axes[row_idx, 3].set_title(f"SGA {gname.title()} NIR (absent)", fontsize=9, color=gcol)

plt.suptitle(
    "Dynamic Range: FTW Sentinel-2 T1  |  SGA Drone  |  SGA Satellite\n"
    "(pooled across all samples per group)",
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_dynamic_range.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. JPG → Synthetic 8-Band GeoTIFF

**Channel layout:** `[R, G, B, G(NIR), R, G, B, G(NIR)]`
- Green is duplicated as synthetic NIR.
- The 4-band block is duplicated to simulate two temporal layers.
- RGB `[0,255]` → uint16 `[0,3000]` to match Sentinel-2 DN range.
- Dummy EPSG:32632 CRS with 10 m pixels assigned.

In [ ]:
def jpg_to_synthetic_geotiff(jpg_path: str, out_tif: str) -> dict:
    """Convert a JPEG to a synthetic 8-band GeoTIFF compatible with the FTW model."""
    img = np.array(Image.open(jpg_path).convert("RGB"))  # (H, W, 3) uint8
    h, w = img.shape[:2]

    # Scale [0,255] → [0,3000] uint16 to match Sentinel-2 L2A DN range
    scaled = (img.astype(np.float32) / 255.0 * 3000).astype(np.uint16)
    r   = scaled[:, :, 0]
    g   = scaled[:, :, 1]
    b   = scaled[:, :, 2]
    nir = g.copy()  # synthetic NIR = Green

    # 8-band: [R,G,B,NIR] duplicated for 2nd temporal layer
    out_array = np.stack([r, g, b, nir, r, g, b, nir], axis=0)  # (8, H, W)

    transform = from_origin(ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, PIXEL_SIZE_M)
    profile = {
        "driver": "GTiff", "dtype": "uint16",
        "width": w, "height": h, "count": 8,
        "crs": TARGET_CRS, "transform": transform, "compress": "lzw",
    }
    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_array)

    return {"height": h, "width": w, "original_rgb": img}


print("Converting JPEG test images to synthetic GeoTIFFs...")
geotiff_paths  = {}   # stem → synthetic tif path
image_metadata = {}   # stem → {height, width, original_rgb}

for group_name, group_info in SGA_GROUPS.items():
    for fname in group_info["files"]:
        stem     = os.path.splitext(fname)[0]
        jpg_path = os.path.join(group_info["dir"], fname)
        tif_path = os.path.join(OUTPUT_DIR, f"{stem}_synthetic.tif")
        meta = jpg_to_synthetic_geotiff(jpg_path, tif_path)
        geotiff_paths[stem]  = tif_path
        image_metadata[stem] = meta
        print(f"  [{group_name}] {fname} → {os.path.basename(tif_path)}  [{meta['height']}×{meta['width']} px]")

print("Done.")

## 4. Inference

Calls `ftw_tools.inference.inference.run()` directly (the same function the CLI wraps)
for each synthetic GeoTIFF. Satellite and drone images are handled differently:

| Property | Satellite | Drone |
|---|---|---|
| Resolution | Native (unchanged) | Resized to 256×256 px |
| Patch size | 256 (FTW standard, sliding window) | 256 (single patch, no tiling) |
| Strategy | Slide over the full image | Whole image fits in one patch |

- `num_workers=0` — Windows Jupyter multiprocessing safety.
- `gpu=None` → falls back to CPU if no CUDA device is available.
- `image_metadata` is updated for drone images to reflect resized dimensions so that downstream visualization cells stay consistent.

In [ ]:
from rasterio.enums import Resampling

FTW_PATCH_SIZE = 256  # FTW standard patch size

pred_tif_paths = {}  # stem → prediction tif path

for stem, tif_path in geotiff_paths.items():
    group    = sga_samples[stem]["group"]
    meta     = image_metadata[stem]
    h, w     = meta["height"], meta["width"]
    pred_tif = os.path.join(OUTPUT_DIR, f"{stem}_pred.tif")

    if group == "satellite":
        # Keep native resolution; slide a 256×256 window (same as FTW training)
        inf_tif    = tif_path
        patch_size = FTW_PATCH_SIZE
        print(f"\n[satellite] {stem}  [{h}×{w} px]  patch_size={patch_size}  (sliding window)")

    else:
        # Drone: resize the whole image to one FTW patch (256×256) and infer in one shot
        inf_tif    = os.path.join(OUTPUT_DIR, f"{stem}_resized.tif")
        patch_size = FTW_PATCH_SIZE
        print(f"\n[drone] {stem}  [{h}×{w}] → [{patch_size}×{patch_size} px]  (single patch)")
        with rasterio.open(tif_path) as src:
            data = src.read(
                out_shape=(src.count, patch_size, patch_size),
                resampling=Resampling.bilinear,
            )
            profile = src.profile.copy()
            profile.update(
                width=patch_size, height=patch_size,
                transform=from_origin(ORIGIN_X, ORIGIN_Y, PIXEL_SIZE_M, PIXEL_SIZE_M),
            )
        with rasterio.open(inf_tif, "w", **profile) as dst:
            dst.write(data)
        # Update metadata so downstream visualization cells use consistent dimensions
        image_metadata[stem]["height"] = patch_size
        image_metadata[stem]["width"]  = patch_size
        image_metadata[stem]["original_rgb"] = np.array(
            Image.fromarray(image_metadata[stem]["original_rgb"])
                 .resize((patch_size, patch_size), Image.BILINEAR)
        )

    inference_run(
        input=inf_tif,
        model=CHECKPOINT,
        out=pred_tif,
        resize_factor=1,
        gpu=None,
        patch_size=patch_size,
        batch_size=1,
        num_workers=0,
        padding=None,
        overwrite=True,
        mps_mode=False,
        save_scores=False,
    )
    pred_tif_paths[stem] = pred_tif
    print(f"  Saved: {pred_tif}")

print(f"\nInference complete for {len(pred_tif_paths)} images.")

## 5. Prediction Mask Visualization

Shows the original RGB, raw prediction mask, and a colour overlay for all images.

Colour legend: **grey** = background (0), **green** = field interior (1), **red** = boundary (2)

In [ ]:
n = len(pred_tif_paths)
fig, axes = plt.subplots(n, 3, figsize=(18, 6 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for row, (stem, pred_tif) in enumerate(pred_tif_paths.items()):
    orig_rgb = image_metadata[stem]["original_rgb"]  # (H, W, 3) uint8

    with rasterio.open(pred_tif) as src:
        pred = src.read(1)  # (H, W), values {0, 1, 2}

    overlay = np.zeros((*pred.shape, 4), dtype=np.float32)
    overlay[pred == 1] = [0.3, 0.8, 0.3, 0.45]
    overlay[pred == 2] = [0.9, 0.2, 0.2, 0.70]

    axes[row, 0].imshow(orig_rgb)
    axes[row, 0].set_title(f"{stem}\nOriginal RGB")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(pred, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest")
    axes[row, 1].set_title(f"{stem}\nPrediction Mask")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(orig_rgb)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"{stem}\nPrediction Overlay")
    axes[row, 2].axis("off")

fig.legend(handles=LEGEND_PATCHES, loc="lower center", ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, "predictions_overview.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Post-Processing

Two stages applied sequentially (`chain=True`):

1. **Morphological opening** (`kernel_size=2`): erode then dilate the field class to
   remove small noise blobs.
2. **Watershed segmentation** (`kernel_size=5`): distance transform + local maxima seeds
   to separate touching field instances into individually labeled regions (uint16).

Both functions are imported from `plot_utils.py` and operate on TIF files.

In [ ]:
MORPH_KERNEL     = 2
WATERSHED_KERNEL = 5

morph_tif_paths     = {}  # stem → morph tif path
watershed_tif_paths = {}  # stem → watershed tif path

n = len(pred_tif_paths)
fig, axes = plt.subplots(n, 4, figsize=(22, 6 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for row, (stem, pred_tif) in enumerate(pred_tif_paths.items()):
    morph_tif     = os.path.join(OUTPUT_DIR, f"{stem}_morph.tif")
    watershed_tif = os.path.join(OUTPUT_DIR, f"{stem}_watershed.tif")

    # chain=True: watershed runs on morph output
    morph = morphological_opening(pred_tif,  morph_tif,     kernel_size=MORPH_KERNEL)
    wshed = watershed_segmentation(morph_tif, watershed_tif, kernel_size=WATERSHED_KERNEL)

    morph_tif_paths[stem]     = morph_tif
    watershed_tif_paths[stem] = watershed_tif

    with rasterio.open(pred_tif) as src:
        pred = src.read(1)

    orig_rgb = image_metadata[stem]["original_rgb"]

    panels = [
        (orig_rgb, f"{stem}\nOriginal RGB",              {}),
        (pred,     "Raw Prediction",                     {"cmap": CMAP, "vmin": 0, "vmax": 2, "interpolation": "nearest"}),
        (morph,    f"Morph Opening (k={MORPH_KERNEL})",  {"cmap": CMAP, "vmin": 0, "vmax": 2, "interpolation": "nearest"}),
        (wshed,    f"Watershed (k={WATERSHED_KERNEL})",  {"interpolation": "nearest"}),
    ]
    for col, (data, title, kw) in enumerate(panels):
        axes[row, col].imshow(data, **kw)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")

fig.legend(handles=LEGEND_PATCHES, loc="lower center", ncol=3, fontsize=11)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, "postprocessing_results.png"), dpi=150, bbox_inches="tight")
plt.show()

## Reflections
The model failed to detect fields in the test RGB images (either from drone or satellite). 
This is due to domain shifts: 
- different sensor characteristics, 
- no NIR channel, 
- single temporal layer.